In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from skeleton.data_loading import *
import pandas as pd
import numpy as np


In [5]:
X, y = load_dataset("credit-g")
X[80:90]

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker
80,no checking,24,existing paid,radio/tv,5943.0,no known savings,<1,1,female div/dep/mar,none,1,car,44,none,own,2,skilled,1,yes,yes
81,no checking,15,existing paid,radio/tv,1213.0,500<=X<1000,>=7,4,male single,none,3,life insurance,47,stores,own,1,skilled,1,yes,yes
82,no checking,18,existing paid,business,1568.0,100<=X<500,1<=X<4,3,female div/dep/mar,none,4,life insurance,24,none,rent,1,unskilled resident,1,none,yes
83,<0,24,existing paid,other,1755.0,<100,>=7,4,female div/dep/mar,guarantor,4,real estate,58,none,own,1,unskilled resident,1,yes,yes
84,<0,10,existing paid,radio/tv,2315.0,<100,>=7,3,male single,none,4,real estate,52,none,own,1,unskilled resident,1,none,yes
85,no checking,12,critical/other existing credit,business,1412.0,<100,1<=X<4,4,female div/dep/mar,guarantor,2,real estate,29,none,own,2,high qualif/self emp/mgmt,1,yes,yes
86,0<=X<200,18,critical/other existing credit,furniture/equipment,1295.0,<100,<1,4,female div/dep/mar,none,1,life insurance,27,none,own,2,skilled,1,none,yes
87,0<=X<200,36,existing paid,education,12612.0,100<=X<500,1<=X<4,1,male single,none,4,no known property,47,none,for free,1,skilled,2,yes,yes
88,<0,18,existing paid,new car,2249.0,100<=X<500,4<=X<7,4,male single,none,3,car,30,none,own,1,high qualif/self emp/mgmt,2,yes,yes
89,<0,12,no credits/all paid,repairs,1108.0,<100,4<=X<7,4,male single,none,3,real estate,28,none,own,2,skilled,1,none,yes


(Clump_Thickness           4.0
 Cell_Size_Uniformity      8.0
 Cell_Shape_Uniformity     6.0
 Marginal_Adhesion         4.0
 Single_Epi_Cell_Size      3.0
 Bare_Nuclei               4.0
 Bland_Chromatin          10.0
 Normal_Nucleoli           6.0
 Mitoses                   1.0
 Name: 697, dtype: float64,
 Clump_Thickness           4.0
 Cell_Size_Uniformity      8.0
 Cell_Shape_Uniformity     8.0
 Marginal_Adhesion         5.0
 Single_Epi_Cell_Size      4.0
 Bare_Nuclei               5.0
 Bland_Chromatin          10.0
 Normal_Nucleoli           4.0
 Mitoses                   1.0
 Name: 698, dtype: float64)

In [ ]:
X.duplicated

np.int64(236)

In [4]:
def explore_datasets_by_name(names: list[str], cache_dir: str = "data_cache") -> list[dict]:
    """Loads datasets by name and inspects instances, duplicates, missing values, dtypes, cardinality, and class imbalance."""
    reports = []
    for name in names:
        X, y = load_dataset(name, cache_dir)
        reports.append({
            "dataset": name,
            "instances": len(X),
            "duplicates": X.duplicated().sum(),
            "dtypes": X.dtypes.value_counts().to_dict(),
            "missing_per_col": X.isna().sum().to_dict(),
            "cardinality": X.nunique().to_dict(),
            "class_imbalance": pd.Series(y).value_counts(normalize=True).round(4).to_dict(),
        })
    return reports

In [5]:
DATASETS = {
    "breast-w": 15,
    "credit-g": 31,
    "phoneme": 1489,
    "electricity": 151,
    "covertype": 1596,
}

reports = explore_datasets_by_name(list(DATASETS.keys()))

In [ ]:
for 

In [10]:
reports

[{'dataset': 'breast-w',
  'instances': 699,
  'duplicates': np.int64(236),
  'dtypes': {dtype('float64'): 9},
  'missing_per_col': {'Clump_Thickness': 0,
   'Cell_Size_Uniformity': 0,
   'Cell_Shape_Uniformity': 0,
   'Marginal_Adhesion': 0,
   'Single_Epi_Cell_Size': 0,
   'Bare_Nuclei': 16,
   'Bland_Chromatin': 0,
   'Normal_Nucleoli': 0,
   'Mitoses': 0},
  'cardinality': {'Clump_Thickness': 10,
   'Cell_Size_Uniformity': 10,
   'Cell_Shape_Uniformity': 10,
   'Marginal_Adhesion': 10,
   'Single_Epi_Cell_Size': 10,
   'Bare_Nuclei': 10,
   'Bland_Chromatin': 10,
   'Normal_Nucleoli': 10,
   'Mitoses': 9},
  'class_imbalance': {0: 0.6552, 1: 0.3448}},
 {'dataset': 'credit-g',
  'instances': 1000,
  'duplicates': np.int64(0),
  'dtypes': {dtype('uint8'): 6,
   CategoricalDtype(categories=['<0', '0<=X<200', '>=200', 'no checking'], ordered=True, categories_dtype=str): 1,
   CategoricalDtype(categories=['no credits/all paid', 'all paid', 'existing paid',
                     'delayed 

In [6]:
def check_splits_and_preprocess(name: str, cache_dir: str = "data_cache") -> dict:
    """Splits, preprocesses data, and verifies class distributions across all splits."""
    splits = load_and_split(name, cache_dir=cache_dir)
    X_train_proc, X_valid_proc = prepare_data(splits)
    
    preprocessor = make_preprocessor(splits.X_train).fit(splits.X_train)
    X_test_proc = np.asarray(preprocessor.transform(splits.X_test), dtype=np.float32)
    
    dist = lambda y: pd.Series(y).value_counts(normalize=True).round(4).to_dict()
    
    return {
        "dataset": name,
        "processed_shapes": {
            "X_train": X_train_proc.shape,
            "X_valid": X_valid_proc.shape,
            "X_test": X_test_proc.shape,
        },
        "class_distributions": {
            "train": dist(splits.y_train),
            "valid": dist(splits.y_valid),
            "test": dist(splits.y_test),
        }
    }

In [8]:
for ds in list(DATASETS.keys()):
    r = check_splits_and_preprocess(ds)
    print(r)
    

{'dataset': 'breast-w', 'processed_shapes': {'X_train': (419, 9), 'X_valid': (140, 9), 'X_test': (140, 9)}, 'class_distributions': {'train': {0: 0.6539, 1: 0.3461}, 'valid': {0: 0.6571, 1: 0.3429}, 'test': {0: 0.6571, 1: 0.3429}}}
{'dataset': 'credit-g', 'processed_shapes': {'X_train': (600, 20), 'X_valid': (200, 20), 'X_test': (200, 20)}, 'class_distributions': {'train': {1: 0.7, 0: 0.3}, 'valid': {1: 0.7, 0: 0.3}, 'test': {1: 0.7, 0: 0.3}}}
{'dataset': 'phoneme', 'processed_shapes': {'X_train': (3242, 5), 'X_valid': (1081, 5), 'X_test': (1081, 5)}, 'class_distributions': {'train': {0: 0.7064, 1: 0.2936}, 'valid': {0: 0.7068, 1: 0.2932}, 'test': {0: 0.7068, 1: 0.2932}}}
{'dataset': 'electricity', 'processed_shapes': {'X_train': (27186, 8), 'X_valid': (9063, 8), 'X_test': (9063, 8)}, 'class_distributions': {'train': {0: 0.5755, 1: 0.4245}, 'valid': {0: 0.5754, 1: 0.4246}, 'test': {0: 0.5754, 1: 0.4246}}}
{'dataset': 'covertype', 'processed_shapes': {'X_train': (348606, 54), 'X_valid': 